# 04 - Filtre de Régime de Marché

## Idée

Le momentum fonctionne bien en marché haussier mais souffre lors des retournements. Un filtre de régime permet de réduire l'exposition pendant les périodes défavorables.

**Approche SMA200:**
- **Risk-On**: S&P 500 > SMA200 → Exposition normale
- **Risk-Off**: S&P 500 < SMA200 → Exposition réduite (50%)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

from src.data_loader import load_universe_data, load_benchmark_data, load_benchmark_daily
from src.backtest import run_backtest
from src.regime_filter import compute_regime_filter, analyze_regime_performance
from src.metrics import compute_metrics
from src.config import get_config

config = get_config()

In [ ]:
prices = load_universe_data(verbose=False)
benchmark_monthly = load_benchmark_data(verbose=False)
benchmark_daily = load_benchmark_daily(verbose=False)

print(f"Données daily: {benchmark_daily.index[0].date()} - {benchmark_daily.index[-1].date()}")

## Calcul du filtre SMA200

In [ ]:
regime = compute_regime_filter(benchmark_daily, config)

risk_on_pct = regime.mean()
print(f"Pourcentage du temps en Risk-On: {risk_on_pct:.1%}")
print(f"Pourcentage du temps en Risk-Off: {1-risk_on_pct:.1%}")

In [ ]:
# Visualisation
from src.regime_filter import compute_sma

spy_price = benchmark_daily.iloc[:, 0]
sma200 = compute_sma(spy_price, 200)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Prix et SMA
axes[0].plot(spy_price.index, spy_price, label='S&P 500', linewidth=1)
axes[0].plot(sma200.index, sma200, label='SMA 200', linewidth=1.5, linestyle='--')
axes[0].set_title('S&P 500 vs SMA200')
axes[0].set_ylabel('Prix')
axes[0].legend()

# Régime
regime_daily = (spy_price > sma200).astype(int)
axes[1].fill_between(regime_daily.index, 0, regime_daily, alpha=0.5, 
                     color='green', label='Risk-On', step='pre')
axes[1].fill_between(regime_daily.index, 0, 1-regime_daily, alpha=0.5, 
                     color='red', label='Risk-Off', step='pre')
axes[1].set_title('Régime de marché')
axes[1].set_ylabel('Régime')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Risk-Off', 'Risk-On'])
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## Performance par régime

In [ ]:
bench_returns = benchmark_monthly.pct_change().iloc[:, 0]
regime_perf = analyze_regime_performance(regime, bench_returns)

print("Performance du S&P 500 par régime:")
regime_perf

## Backtest avec filtre de régime

In [ ]:
# Sans filtre
config.PORTFOLIO_TYPE = 'long_short'
results_no_filter = run_backtest(prices, benchmark_monthly, config, regime_signal=None)

# Avec filtre
results_with_filter = run_backtest(prices, benchmark_monthly, config, regime_signal=regime)

print("Backtests terminés.")

In [ ]:
# Comparaison
fig, ax = plt.subplots(figsize=(14, 7))

results_no_filter.cumulative_returns.plot(ax=ax, label='Sans filtre', linewidth=2)
results_with_filter.cumulative_returns.plot(ax=ax, label='Avec filtre régime', linewidth=2)
results_no_filter.cumulative_benchmark.plot(ax=ax, label='S&P 500', 
                                            linewidth=2, color='gray', linestyle='--')

ax.set_title('Momentum L/S: Impact du filtre de régime')
ax.set_ylabel('Valeur ($1 initial)')
ax.set_yscale('log')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
metrics_no_filter = compute_metrics(
    results_no_filter.strategy_returns,
    results_no_filter.cumulative_returns,
    results_no_filter.turnover,
    results_no_filter.transaction_costs
)

metrics_with_filter = compute_metrics(
    results_with_filter.strategy_returns,
    results_with_filter.cumulative_returns,
    results_with_filter.turnover,
    results_with_filter.transaction_costs
)

comparison = pd.DataFrame({
    'Sans filtre': [
        f"{metrics_no_filter.cagr:.2%}",
        f"{metrics_no_filter.volatility:.2%}",
        f"{metrics_no_filter.sharpe_ratio:.2f}",
        f"{metrics_no_filter.max_drawdown:.2%}",
        f"{metrics_no_filter.calmar_ratio:.2f}",
    ],
    'Avec filtre': [
        f"{metrics_with_filter.cagr:.2%}",
        f"{metrics_with_filter.volatility:.2%}",
        f"{metrics_with_filter.sharpe_ratio:.2f}",
        f"{metrics_with_filter.max_drawdown:.2%}",
        f"{metrics_with_filter.calmar_ratio:.2f}",
    ],
}, index=['CAGR', 'Volatilité', 'Sharpe', 'Max Drawdown', 'Calmar'])

print("Impact du filtre de régime:")
comparison

## Analyse des drawdowns

In [ ]:
from src.backtest import compute_drawdowns

dd_no_filter = compute_drawdowns(results_no_filter.cumulative_returns)
dd_with_filter = compute_drawdowns(results_with_filter.cumulative_returns)

fig, ax = plt.subplots(figsize=(14, 5))

dd_no_filter.plot(ax=ax, label='Sans filtre', alpha=0.8)
dd_with_filter.plot(ax=ax, label='Avec filtre', alpha=0.8)

ax.axhline(-0.2, color='red', linestyle='--', alpha=0.5)
ax.set_title('Drawdowns: Impact du filtre')
ax.set_ylabel('Drawdown')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Amélioration du Max Drawdown: {metrics_no_filter.max_drawdown - metrics_with_filter.max_drawdown:.2%}")

## Conclusions

**Le filtre SMA200:**
1. Réduit la volatilité et les drawdowns
2. Améliore le Sharpe et le Calmar ratio
3. Simple à implémenter et à expliquer
4. Pas de risque d'overfitting (règle fixe)

**Limites:**
- Signal retardé (trend-following)
- Peut générer des faux signaux en marché latéral
- Le choix de 200 jours est une convention, pas optimisé